In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from pathlib import Path
import openslide

import sys
sys.path.append("/workspaces/TRIDENT")

from trident.wsi_objects.entropy_segmentation import entropy_mask

In [ ]:
from skimage.filters import threshold_otsu
from skimage.color import rgb2hsv
from skimage.transform import resize
from skimage.exposure import equalize_adapthist, rescale_intensity

def thumbnail_segmentation(
    slide_path: str,
):
    mask = entropy_mask(slide_path)
    height, width = mask.shape

    # Load the WSI
    with openslide.OpenSlide(slide_path) as slide:
        thumbnail = slide.get_thumbnail((width, height))

    # Ensure that the thumbnail is actually the correct size
    if thumbnail.size != (width, height):
        thumbnail = thumbnail.resize((width, height))

    masked_thumbnail = np.array(thumbnail)

    # Convert to numpy array
    # Set everything in the thumbnail to white where mask is 0
    masked_thumbnail[mask == 0] = [255, 255, 255]

    # Convert to HSV
    masked_hue_channel = rgb2hsv(masked_thumbnail)[:, :, 0]

    # Threshold 
    hue_threshold = threshold_otsu(masked_hue_channel)
    hue_mask = masked_hue_channel > hue_threshold

    # resize back to mask size
    final_mask = resize(hue_mask, (height, width), anti_aliasing=False)

    return final_mask


def thumbnail_segmentation_verbose(
    slide_path: str,
):
    factor = 1
    fig, axs = plt.subplots(2, 2, figsize=(20, 15))
    axs = axs.flatten()
    mask = entropy_mask(slide_path)
    mask_height, mask_width = mask.shape
    print(f"mask shape: {mask.shape}")
    height = mask_height * factor
    width = mask_width * factor

    
    upscaled_mask = mask.copy()
    if factor > 1:
        # Resize the mask to match the thumbnail size
        upscaled_mask = resize(mask.copy(), (height, width), anti_aliasing=False)
        print(f"upscaled mask shape: {upscaled_mask.shape}")

    axs[0].imshow((mask * 255).astype(np.uint8), cmap="gray")
    axs[0].set_title("Entropy Mask")
    axs[0].axis("off")

    # Load the WSI
    with openslide.OpenSlide(slide_path) as slide:
        thumbnail = slide.get_thumbnail((width, height))
        print(f"thumbnail size: {thumbnail.size}")

        # Ensure that the thumbnail is actually the correct size
        if thumbnail.size != (width, height):
            thumbnail = thumbnail.resize((width, height))

        axs[1].imshow(thumbnail)
        axs[1].set_title("Thumbnail")
        axs[1].axis("off")

        # Convert to numpy array
        masked_thumbnail = np.array(thumbnail)
        # Set everything in the thumbnail to white where mask is 0
        masked_thumbnail[upscaled_mask == 0] = [255, 255, 255]

        axs[2].imshow(masked_thumbnail)
        axs[2].set_title("Masked Thumbnail")
        axs[2].axis("off")

    # Convert to HSV
    masked_thumbnail_hsv = rgb2hsv(masked_thumbnail)
    masked_hue_channel = masked_thumbnail_hsv[:, :, 0]

    # Threshold 
    hue_threshold = threshold_otsu(masked_hue_channel)
    hue_mask = masked_hue_channel > hue_threshold

    # resize back to mask size
    final_mask = resize(hue_mask, (mask_height, mask_width), anti_aliasing=False)
    # Convert to uint8 for display
    final_mask = (final_mask * 255).astype(np.uint8)
    axs[3].imshow(final_mask, cmap="gray")
    axs[3].set_title("Combined Mask")
    axs[3].axis("off")
    fig.tight_layout()
    fig.show()
    # return final_mask


In [ ]:
examples = [Path("/mnt/oskar/slides/n2020001261t1-a-3_190650.svs"),
    Path("/mnt/oskar/slides/h2019018560t5-a-2_161036.svs"),
    Path("/mnt/oskar/slides/n2022001164t1-a-3_124229.svs"),
    Path("/mnt/oskar/slides/n2023001964t3-a-4_113847.svs"),
    Path("/mnt/oskar/slides/h2021014634t2-a-5_023026.svs"),
]

segment_again = [Path("/mnt/oskar/slides/f2024001766t1-a-6_130955.svs"),
Path("/mnt/oskar/slides/h2019018560t5-a-2_161036.svs"),
Path("/mnt/oskar/slides/h2022003279t1-a-3_032528.svs"),
Path("/mnt/oskar/slides/h2022005119t1-a-4_093353.svs"),
Path("/mnt/oskar/slides/h2022007825t1-a-8_233639.svs"),
Path("/mnt/oskar/slides/h2023010049t1-a-5_111127.svs"),
Path("/mnt/oskar/slides/h2024002581t11-a-3_121401.svs"),
Path("/mnt/oskar/slides/h2024007715t2-a-2_102433.svs"),
Path("/mnt/oskar/slides/h2024012704t1-a-2_110442.svs"),
Path("/mnt/oskar/slides/h2024017094t2-a-3_132305.svs"),
Path("/mnt/oskar/slides/n2020001261t1-a-3_190650.svs"),
Path("/mnt/oskar/slides/n2022001164t1-a-3_124229.svs"),
Path("/mnt/oskar/slides/n2023001964t3-a-4_113847.svs")]
missing = [Path("/mnt/oskar/slides/e2024000032t1-a-4_143250.svs")]

# /mnt/oskar/contours/75964808-b9d5-a7ba-6284-893e2c7f441f_183742.jpg
# /mnt/oskar/contours/e2024000005t1-a-3_105708.jpg
# /mnt/oskar/contours/f2024001269t1-a-3_114359.jpg

# fig, axs = plt.subplots(2, 14, figsize=(140, 10))
# for i, slide in enumerate(segment_again + missing):
#     mask = thumbnail_segmentation(slide)
#     axs[0, i].imshow(mask, cmap="gray")
#     axs[0, i].set_title("Thumbnail Segmentation")
#     axs[0, i].axis("off")

#     e_mask = entropy_mask(slide)
#     axs[1, i].imshow(e_mask, cmap="gray")
#     axs[1, i].set_title("Entropy Segmentation")
#     axs[1, i].axis("off")


# fig, axs = plt.subplots(2, 5, figsize=(50, 10))

# for i, slide in enumerate(examples):
#     mask = thumbnail_segmentation(slide)
#     axs[0, i].imshow(mask, cmap="gray")
#     axs[0, i].set_title("Thumbnail Segmentation")
#     axs[0, i].axis("off")

#     e_mask = entropy_mask(slide)
#     axs[1, i].imshow(e_mask, cmap="gray")
#     axs[1, i].set_title("Entropy Segmentation")
#     axs[1, i].axis("off")


thumbnail_segmentation_verbose(
    Path("/mnt/oskar/slides/h2025001976t1-a-4_102918.svs"),
)



In [ ]:
import PIL.Image
interesting_examples = [
    Path("/mnt/oskar/slides/h2021014634t2-a-5_023026.svs"), 
    Path("/mnt/oskar/slides/h2021013146t1-a-6_120332.svs"),
    Path("/mnt/oskar/slides/h2023010686t1-a-6_094959.svs"),
    Path("/mnt/oskar/slides/h2025001976t1-a-4_102918.svs"),
]
def show_entropy_mask(slide_path: Path, trident_path: Path):
    fig, ax = plt.subplots(1, 3, figsize=(20, 10))
    e_mask = entropy_mask(slide_path)
    # e_mask = thumbnail_segmentation(slide_path)
    ax[0].imshow(e_mask, cmap="gray")
    ax[0].set_title(slide_path.stem)
    ax[0].axis("off")
    with openslide.OpenSlide(slide_path) as slide:
        height, width = e_mask.shape
        thumbnail = slide.get_thumbnail((width, height))
        thumbnail = thumbnail.resize((width, height))
        ax[1].imshow(thumbnail)
        ax[1].axis("off")
        ax[1].set_title("Thumbnail")

    trident_thumbnail = PIL.Image.open(trident_path)
    trident_thumbnail = trident_thumbnail.resize((width, height))
    ax[2].imshow(trident_thumbnail)
    ax[2].axis("off")
    ax[2].set_title("Trident Attempt")
    plt.show()


for slide in interesting_examples:
    show_entropy_mask(slide, Path("/mnt/oskar/contours") / (slide.stem + ".jpg"))